In [1]:
import requests
import json
from datetime import date

API_BASE = "https://statsapi.mlb.com"

def get_todays_games():
    today = date.today().isoformat()
    url = f"{API_BASE}/api/v1/schedule/games/?sportId=1&date={today}"
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    data = r.json()
    if not data.get("dates"):
        return []
    return data["dates"][0].get("games", [])

def fetch_live(gamePk):
    urls = [
        f"{API_BASE}/api/v1.1/game/{gamePk}/feed/live",
        f"{API_BASE}/api/v1/game/{gamePk}/feed/live"
    ]
    for url in urls:
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                return r.json(), url
        except:
            pass
    return None, None

# -----------------------------
# MAIN DEBUG LOGIC
# -----------------------------
games = get_todays_games()

target = None
for g in games:
    away = g["teams"]["away"]["team"]["name"]
    home = g["teams"]["home"]["team"]["name"]
    if ("Yankees" in away or "Yankees" in home) and ("Royals" in away or "Royals" in home):
        target = g
        break

if not target:
    print("No NYY vs KC game found today.")
    exit()

gamePk = target["gamePk"]
print(f"Found NYY vs KC gamePk: {gamePk}")

live_json, used_url = fetch_live(gamePk)

if not live_json:
    print("Could not fetch live data.")
    exit()

print(f"Fetched live data from: {used_url}\n")

# Extract inning info
ls = live_json.get("liveData", {}).get("linescore", {})

print("----- LINESCORE DEBUG -----")
print("currentInning:", ls.get("currentInning"))
print("inningState:", ls.get("inningState"))
print("outs:", ls.get("outs"))

teams = ls.get("teams", {})
print("away runs:", teams.get("away", {}).get("runs"))
print("home runs:", teams.get("home", {}).get("runs"))

print("\n----- FULL LINESCORE JSON -----")
print(json.dumps(ls, indent=2))

# Uncomment to dump the entire live feed:
# print("\n----- FULL LIVE JSON -----")
# print(json.dumps(live_json, indent=2))


Found NYY vs KC gamePk: 824107
Fetched live data from: https://statsapi.mlb.com/api/v1.1/game/824107/feed/live

----- LINESCORE DEBUG -----
currentInning: 2
inningState: Bottom
outs: 1
away runs: 2
home runs: 0

----- FULL LINESCORE JSON -----
{
  "currentInning": 2,
  "currentInningOrdinal": "2nd",
  "inningState": "Bottom",
  "inningHalf": "Bottom",
  "isTopInning": false,
  "scheduledInnings": 9,
  "innings": [
    {
      "num": 1,
      "ordinalNum": "1st",
      "home": {
        "runs": 0,
        "hits": 0,
        "errors": 0,
        "leftOnBase": 0
      },
      "away": {
        "runs": 0,
        "hits": 0,
        "errors": 0,
        "leftOnBase": 0
      }
    },
    {
      "num": 2,
      "ordinalNum": "2nd",
      "home": {
        "hits": 0,
        "errors": 0,
        "leftOnBase": 0
      },
      "away": {
        "runs": 2,
        "hits": 3,
        "errors": 0,
        "leftOnBase": 2
      }
    }
  ],
  "teams": {
    "home": {
      "runs": 0,
      "hits